# Aiko — fine-tuning LFM2.5-VL-1.6B avec Unsloth + LoRA

Ce notebook entraîne un adapter LoRA à partir du dataset local versionné dans data/aiko_sft.jsonl. Aiko est une femme japonaise fictive et adulte, très présente sur Discord, qui répond en français SMS avec des kaomojis, de l'humour et un bloc de réflexion structuré entre <think> et </think>.

Le corpus contient des conversations courtes et longues, un preprompt système complet répété dans chaque exemple, et un split train/eval explicite.


## Principes

- data/aiko_sft.jsonl est la source de vérité : aucune conversation n'est générée en dur dans le notebook.
- data/aiko_system_prompt.txt est le preprompt canonique ; chaque ligne JSONL doit lui correspondre exactement.
- Le dataset conserve le message system pour être autonome. Au moment du collator, il est retiré pour éviter le conflit entre le template ChatML LFM2.5-VL et la normalisation Unsloth ; le même prompt est utilisé à l'inférence.
- Le vision encoder reste gelé : le LoRA apprend surtout la persona, le français SMS, le contexte Discord et le format de raisonnement court.
- Les balises think sont une convention de sortie supervisée, pas une garantie de raisonnement fiable.


In [ ]:
# Colab / Kaggle : exécuter cette cellule une seule fois, puis redémarrer le runtime si demandé.
%pip install -U --no-cache-dir 'unsloth[colab-new]' 'transformers>=5.1' 'datasets>=3.0' 'trl>=0.23'


In [ ]:
import json
import os
import random
from pathlib import Path

import torch
from datasets import Dataset
from unsloth import FastVisionModel, is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTConfig, SFTTrainer

MODEL_ID = 'LiquidAI/LFM2.5-VL-1.6B'
MAX_SEQ_LENGTH = 2048
SEED = 3407
OUTPUT_DIR = 'outputs/aiko-lfm25-vl-lora'
ADAPTER_DIR = 'aiko-lfm25-vl-lora'

DATASET_PATH = Path('data/aiko_sft.jsonl')
SYSTEM_PROMPT_PATH = Path('data/aiko_system_prompt.txt')
if not DATASET_PATH.exists() or not SYSTEM_PROMPT_PATH.exists():
    raise FileNotFoundError(
        'Lance le notebook depuis la racine du dépôt : data/aiko_sft.jsonl est requis.'
    )

random.seed(SEED)
torch.manual_seed(SEED)
print(f'torch={torch.__version__}')
print(f'cuda={torch.cuda.is_available()} | bf16={is_bf16_supported()}')


## 1. Chargement et contrôle du dataset local

Le format Git est volontairement lisible : messages texte classiques avec un message system complet. Le collator vision reçoit ensuite le même contenu sous forme de blocs textuels, sans le message system d'entraînement.


In [ ]:
BASE_SYSTEM_PROMPT = SYSTEM_PROMPT_PATH.read_text(encoding='utf-8').strip()

with DATASET_PATH.open(encoding='utf-8') as stream:
    raw_rows = [json.loads(line) for line in stream if line.strip()]

assert raw_rows, 'Le dataset est vide.'
assert len({row['id'] for row in raw_rows}) == len(raw_rows)
assert {row['split'] for row in raw_rows} == {'train', 'eval'}

for row in raw_rows:
    messages = row['messages']
    system_content = messages[0].get('content', '')
    assert all(marker in BASE_SYSTEM_PROMPT and marker in system_content for marker in ('Tu es Aiko', 'Raisonnement visible', 'Frontières', 'Sécurité'))
    assert messages[-1]['role'] == 'assistant'
    for index, message in enumerate(messages[1:]):
        expected_role = 'user' if index % 2 == 0 else 'assistant'
        assert message['role'] == expected_role
        assert message['content'].strip()
        if message['role'] == 'assistant':
            assert message['content'].startswith('<think>')

def to_vision_example(row):
    # Le system prompt reste vérifié dans le dataset, mais est injecté à l'inférence.
    return {
        'messages': [
            {
                'role': message['role'],
                'content': [{'type': 'text', 'text': message['content']}],
            }
            for message in row['messages'][1:]
        ]
    }

SYSTEM_PROMPT = next(row['messages'][0]['content'] for row in raw_rows if row['category'] == '01_discord')
print(f'system_variants={len({row["messages"][0]["content"] for row in raw_rows})}')

train_dataset = Dataset.from_list([
    to_vision_example(row) for row in raw_rows if row['split'] == 'train'
])
eval_dataset = Dataset.from_list([
    to_vision_example(row) for row in raw_rows if row['split'] == 'eval'
])

print(f'conversations={len(raw_rows)}')
print(f'train={len(train_dataset)} | eval={len(eval_dataset)}')
print(f'assistant_turns={sum(sum(message["role"] == "assistant" for message in row["messages"]) for row in raw_rows)}')


In [ ]:
from pprint import pprint

pprint(raw_rows[0])
assert train_dataset[0]['messages'][0]['content'][0]['type'] == 'text'
assert '<think>' in train_dataset[0]['messages'][1]['content'][0]['text']


## 2. Chargement du modèle et ajout du LoRA

Le checkpoint natif Liquid AI est utilisé plutôt qu'un GGUF. Un rank 16 est adapté à une première persona LoRA ; augmente-le seulement après avoir enrichi le corpus et défini une vraie évaluation.


In [ ]:
model, processor = FastVisionModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

FastVisionModel.for_training(model)
model.print_trainable_parameters()


## 3. Collator et SFT

Le template ChatML LFM2.5 utilise les marqueurs im_start user et im_start assistant. Le mode train_on_responses_only évite de calculer la loss sur les messages utilisateur.


In [ ]:
data_collator = UnslothVisionDataCollator(
    model,
    processor,
    max_seq_length=MAX_SEQ_LENGTH,
    train_on_responses_only=True,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=processor,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        per_device_train_batch_size=2,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        warmup_ratio=0.1,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        logging_steps=1,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=2,
        optim='adamw_8bit',
        weight_decay=0.01,
        max_grad_norm=1.0,
        bf16=is_bf16_supported(),
        fp16=not is_bf16_supported(),
        gradient_checkpointing=True,
        seed=SEED,
        output_dir=OUTPUT_DIR,
        report_to='none',
        remove_unused_columns=False,
        dataset_text_field='',
        dataset_kwargs={'skip_prepare_dataset': True},
        max_length=MAX_SEQ_LENGTH,
    ),
)


In [ ]:
# Pour un smoke test GPU, remplace temporairement num_train_epochs par max_steps=30.
trainer_stats = trainer.train()
print(trainer_stats.metrics)


## 4. Sauvegarde de l'adapter

Seul le LoRA est sauvegardé : le résultat reste léger et réutilisable avec le checkpoint de base.


In [ ]:
os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)
print(f'Adapter saved to: {ADAPTER_DIR}')


## 5. Test d'inférence

Le preprompt complet est réinjecté ici. Aiko doit produire un court bloc think puis une réponse SMS ; la température reste modérée pour conserver la cohérence du personnage.


In [ ]:
FastVisionModel.for_inference(model)

def generate_aiko(user_text, max_new_tokens=128):
    conversation = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {
            'role': 'user',
            'content': [{'type': 'text', 'text': user_text}],
        },
    ]
    inputs = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors='pt',
        return_dict=True,
    ).to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        min_p=0.15,
        repetition_penalty=1.05,
    )
    generated_ids = outputs[0, inputs['input_ids'].shape[-1]:]
    return processor.decode(generated_ids, skip_special_tokens=True)

print(generate_aiko('t’es encore sur Discord ou tu dors enfin ?'))


## Ajouter des images plus tard

Le corpus actuel entraîne le comportement conversationnel sans embarquer d'assets. Pour un exemple vision, ajoute un bloc image dans le contenu utilisateur, puis garde le même format de réponse assistant. Active le fine-tuning du vision encoder uniquement avec un corpus d'images suffisamment grand et une VRAM adaptée.

Valide toujours le dataset avant de committer :

python3 scripts/validate_aiko_dataset.py
